In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# print(os.getcwd())
# print(os.path.abspath(base_path))


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [2]:
# ls

In [3]:
base_path = "../ablation_results_linear_form"

folders = {

    3: "mmlu_test_eval_service_3_300",
    5: "mmlu_test_eval_service_5_300",
    8: "mmlu_test_eval_service_8_300",
    10: "mmlu_test_eval_service_10_300",
    12: "mmlu_test_eval_service_12_300",
    14: "mmlu_test_eval_service_14_300"

}

In [4]:
data = {}

for k, folder_name in folders.items():

    folder_path = os.path.join(base_path, folder_name)
#     print(folder_path)

    data[k] = []

    for file_name in os.listdir(folder_path):

        if not file_name.endswith(".json"):
            continue

        if "_efficiency" in file_name:
            continue

        file_path = os.path.join(folder_path, file_name)
#         print(file_path)

        with open(file_path, "r") as f:
            rows = json.load(f)
#             print(rows)

        for row in rows:
            row = row.copy()
            row["source_file"] = file_name
            data[k].append(row)

    print(f"K={k}: loaded {len(data[k])} rows")

K=3: loaded 1800 rows
K=5: loaded 2400 rows
K=8: loaded 3300 rows
K=10: loaded 3900 rows
K=12: loaded 4500 rows
K=14: loaded 5100 rows


In [5]:
df_k3 = pd.DataFrame(data[3])
df_k5 = pd.DataFrame(data[5])
df_k8 = pd.DataFrame(data[8])
df_k10 = pd.DataFrame(data[10])
df_k12 = pd.DataFrame(data[12])
df_k14 = pd.DataFrame(data[14])


print("K=3:", df_k3.shape)
print("K=5:", df_k5.shape)
print("K=8:", df_k8.shape)
print("K=10:", df_k10.shape)
print("K=12:", df_k12.shape)
print("K=14:", df_k14.shape)

K=3: (1800, 32)
K=5: (2400, 34)
K=8: (3300, 37)
K=10: (3900, 39)
K=12: (4500, 41)
K=14: (5100, 43)


In [13]:
# combine all K into one table
df_linear = pd.concat([df_k3, df_k5, df_k8, df_k10, df_k12, df_k14], ignore_index=True)
df_linear["K"] = df_linear["total_service_count"]
df_linear["model"] = df_linear["model_name"].str.split("/").str[-1]
df_linear["query_type"] = np.where(df_linear["expected_service"] == "general", "Gen.", "Spec.")
df_linear["is_correct"] = df_linear["is_correct"].astype(float)

# specialist recall and general recall: mean of is_correct, split by query type
recall = df_linear.pivot_table(index=["model", "K"], columns="query_type",
                               values="is_correct", aggfunc="mean")

# accuracy: mean of is_correct over all questions
accuracy = df_linear.pivot_table(index=["model", "K"], values="is_correct", aggfunc="mean")
accuracy.columns = ["Acc."]

linear_results = (100 * accuracy.join(recall))[["Acc.", "Spec.", "Gen."]].round(2)
linear_results

Acc.  Spec.   Gen.
model                K                      
Qwen3-Reranker-4B    3   97.83  98.33  97.33
                     5   94.50  96.40  91.33
                     8   87.73  89.75  82.33
                     10  87.31  88.40  83.67
                     12  83.40  82.92  85.33
                     14  81.59  80.86  85.00
Reason-ModernColBERT 3   92.00  94.00  90.00
                     5   89.25  88.40  90.67
                     8   83.64  79.75  94.00
                     10  80.38  75.50  96.67
                     12  74.20  68.83  95.67
                     14  71.06  65.43  97.33
bge-large-en-v1.5    3   92.50  89.33  95.67
                     5   88.88  86.80  92.33
                     8   82.36  79.62  89.67
                     10  80.38  77.00  91.67
                     12  70.93  68.75  79.67
                     14  67.53  64.71  80.67